# EDA — AI-Based Patient Load Prediction for Government Hospitals
**Project:** Final Year Project — Jaipur Hospital Patient-Flow Dataset (2021–2025)  
**Dataset coverage:** 10 Jaipur hospitals × 1,826 days (1 Jan 2021 – 31 Dec 2025)  

> [!IMPORTANT]
> The weather, bed, occupancy and doctor columns are **simulated** (`_simulated` / `_proxy` in the column names). This must be declared in every report and slide deck.

---

**Files used**
| File | Description |
|------|-------------|
| `patient_flow_from_2021_to_2025.csv` | Per-hospital per-day rows (flow) |
| `ML_model_training_based.csv` | All-10-hospital aggregate per day (agg) |
| `daily_patient_flow_based_on_events.csv` | Calendar / weather flags per day |
| `hospital_and_address_related.csv` | Hospital directory (50+ records) |
| `hospital_and_service_type.csv` | Hospital services & 24×7 emergency status |

**EDA Sections**
0. Setup and load data  
1. Data quality checks  
2. Summary statistics  
3. Trend over time  
4. Weekly, monthly and yearly patterns  
5. Effect of calendar and weather flags  
6. Hospital comparison  
7. Correlations  
8. Autocorrelation  
9. Distributions and outliers  
10. Seasonality decomposition  
11. Bed occupancy and resource analysis  
12. Government vs Private hospital deep-dive  
13. Feature importance preview (mutual information)  
14. Rolling statistics and volatility  
15. Hospital services & 24×7 emergency coverage  
16. EDA summary and feature engineering plan  

## 0. Setup and Load Data

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted")
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    plt.style.use("seaborn-v0_8-whitegrid")

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110

# ── Where are the CSV files? ──────────────────────────────────────────────
# Colab + Drive: uncomment next two lines and set the folder path
# from google.colab import drive; drive.mount("/content/drive")
# DATA = Path("/content/drive/MyDrive/patient-load-project/data")
# Colab with uploaded files  → files sit in /content
# Local / IDE               → notebook in notebooks/, data in ../data
candidates = [Path("../data"), Path("data"), Path("/content"), Path("/mnt/user-data/uploads")]
DATA = next((p for p in candidates if (p / "ML_model_training_based.csv").exists()), None)
assert DATA is not None, "CSV files not found — set DATA manually to the folder holding them"
print("Using data folder:", DATA.resolve())

# ── Constants ─────────────────────────────────────────────────────────────
FLAGS = ["weekend_flag", "monsoon_season_flag", "winter_season_flag",
         "summer_season_flag", "heat_flag", "festival_event_proxy_flag"]
DOW   = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

# ── Loader ────────────────────────────────────────────────────────────────
def load(name, date_col="date"):
    """Read CSV (utf-8-sig strips BOM); parse DD-MM-YYYY dates."""
    df = pd.read_csv(DATA / name, encoding="utf-8-sig")
    if date_col and date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], format="%d-%m-%Y")
    return df

# ── Load all files ────────────────────────────────────────────────────────
flow     = load("patient_flow_from_2021_to_2025.csv")          # 10 hospitals × 1826 days
agg      = load("ML_model_training_based.csv").sort_values("date").reset_index(drop=True)
events   = load("daily_patient_flow_based_on_events.csv")      # calendar/weather flags only
hosp_dir = pd.read_csv(DATA / "hospital_and_address_related.csv", encoding="utf-8-sig")
hosp_svc = pd.read_csv(DATA / "hospital_and_service_type.csv",   encoding="utf-8-sig")

print(f"flow:     {flow.shape}")
print(f"agg:      {agg.shape}")
print(f"events:   {events.shape}")
print(f"hosp_dir: {hosp_dir.shape}")
print(f"hosp_svc: {hosp_svc.shape}")

## 1. Data Quality Checks
**What the files are:**  
- `flow` = per-hospital per-day rows (10 hospitals × 1826 days = 18,260 rows)  
- `agg`  = sum of all 10 hospitals per day (1,826 rows — used for first models)  
- `events` = calendar/weather flag table (no patient counts)

In [ ]:
# ── Shape, date range, missing values, duplicates ──────────────────────────
for name, df in [("hospital-level (flow)", flow), ("aggregate (agg)", agg), ("events", events)]:
    key = ["hospital_id", "date"] if "hospital_id" in df else ["date"]
    print(f"[{name}]")
    print(f"  shape     = {df.shape}")
    if "date" in df.columns:
        print(f"  date range= {df.date.min().date()} → {df.date.max().date()}")
    print(f"  missing   = {int(df.isna().sum().sum())}")
    print(f"  duplicates= {int(df.duplicated(key).sum())}")
    print()

# ── Calendar completeness ──────────────────────────────────────────────────
full = pd.date_range(agg.date.min(), agg.date.max(), freq="D")
print("Missing calendar days (agg):", len(full.difference(agg.date)))

# ── Cross-checks ──────────────────────────────────────────────────────────
total_ok = (flow.total_patient_load == flow.opd_patients + flow.emergency_patients).all()
print("total == opd + emergency (flow):", bool(total_ok))

agg_check = flow.groupby("date")[["opd_patients", "emergency_patients"]].sum()
agg_match  = (agg.set_index("date")[["opd_patients", "emergency_patients"]] == agg_check).all().all()
print("agg equals sum of hospitals:   ", bool(agg_match))

print("\nData types (agg):")
display(agg.dtypes.to_frame("dtype").T)

> **Observation:**  
> The dataset is remarkably clean — **zero missing values** across all three tables and **zero duplicate rows**. All 1,826 calendar days (1 Jan 2021 – 31 Dec 2025) are present with no gaps. The integrity checks pass: `total_patient_load = opd + emergency` holds for every row in `flow`, and the `agg` table exactly equals the per-hospital sums. Because weather, bed and doctor columns are simulated/proxy values (noted in column names), no imputation is needed. The data is ready for EDA without any cleaning step.

## 2. Summary Statistics

In [ ]:
# ── Aggregate-level summary ────────────────────────────────────────────────
print("=== Aggregate (all 10 hospitals combined) ===")
summary = agg.drop(columns=["city"]).describe().T.round(2)
display(summary)

print("\n=== Hospital-level summary ===")
num_cols = ["opd_patients", "emergency_patients", "total_patient_load",
            "admissions", "total_beds_capacity_simulated",
            "beds_available_simulated", "occupancy_pct_simulated",
            "doctors_on_duty_simulated", "temp_mean_proxy_c_simulated",
            "precipitation_proxy_mm_simulated"]
display(flow[num_cols].describe().T.round(2))

> **Observation:**  
> Across all 10 hospitals combined, the **average daily OPD is ~6,028 patients** (std ≈ 662) with a range of 4,503–7,633. Emergency visits average **~461/day** (std ≈ 33), far more stable. Admissions average ~352/day. At the hospital level, bed capacity ranges widely (indicating a mix of large tertiary hospitals and small primary health centres). Occupancy averages ~70–80%, with peaks near 100% for the government tertiary hospitals. Temperature proxy spans ~6°C (winter) to ~43°C (peak summer), and precipitation is mostly 0 mm with occasional monsoon spikes.

## 3. Trend Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

pairs = [
    ("opd_patients",       "#1f77b4", "Daily OPD Patients (all hospitals)"),
    ("emergency_patients", "#d62728", "Daily Emergency Patients (all hospitals)"),
    ("admissions",         "#2ca02c", "Daily Admissions (all hospitals)"),
]
for ax, (col, color, title) in zip(axes, pairs):
    ax.plot(agg.date, agg[col], lw=0.5, alpha=0.4, color=color, label="daily")
    ax.plot(agg.date, agg[col].rolling(30, center=True).mean(),
            lw=2, color="black", label="30-day rolling mean")
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper left", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.show()

# Year-over-year growth
agg["year"] = agg.date.dt.year
print("Year-over-year OPD mean:")
print(agg.groupby("year").opd_patients.mean().round(0))
print("\nYear-over-year Emergency mean:")
print(agg.groupby("year").emergency_patients.mean().round(0))

> **Observation:**  
> The 30-day rolling mean shows a **clear upward trend in OPD** from ~5,500/day in 2021 to ~6,500/day by 2025 — roughly 18% cumulative growth over 5 years. **Emergency visits are far more stable**, showing only marginal growth, which suggests that emergency demand is less elastic than outpatient demand. Admissions track OPD. Prominent annual waves are visible: OPD dips in winter months (Jan–Feb) and surges in spring/summer and again during post-monsoon (Oct–Nov). The daily series is volatile — weekday vs weekend swings are large (~25% drop on weekends).

## 4. Weekly, Monthly and Yearly Patterns

In [ ]:
agg["dow"]   = agg.date.dt.dayofweek   # 0=Mon … 6=Sun
agg["month"] = agg.date.dt.month
agg["year"]  = agg.date.dt.year

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for row, col in enumerate(["opd_patients", "emergency_patients"]):
    # Day-of-week
    ax = axes[row, 0]
    vals = agg.groupby("dow")[col].mean().set_axis(DOW)
    bars = ax.bar(DOW, vals, color=["#d62728" if d in ["Sat", "Sun"] else "#1f77b4" for d in DOW])
    ax.set_title(f"{col} — by day of week", fontsize=10)
    ax.bar_label(bars, fmt="%.0f", fontsize=7)
    ax.set_xlabel("")

    # Month
    ax = axes[row, 1]
    mvals = agg.groupby("month")[col].mean().set_axis(MONTH_NAMES)
    bars2 = ax.bar(MONTH_NAMES, mvals, color="#ff7f0e")
    ax.set_title(f"{col} — by month", fontsize=10)
    ax.bar_label(bars2, fmt="%.0f", fontsize=7)
    ax.set_xlabel("")

    # Year
    ax = axes[row, 2]
    yvals = agg.groupby("year")[col].mean()
    bars3 = ax.bar(yvals.index.astype(str), yvals, color="#2ca02c")
    ax.set_title(f"{col} — by year", fontsize=10)
    ax.bar_label(bars3, fmt="%.0f", fontsize=7)
    ax.set_xlabel("")

plt.suptitle("Temporal Patterns in Patient Load", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Weekday OPD means:", agg.groupby("dow").opd_patients.mean().round(0).set_axis(DOW).to_dict())
print("Weekday Emergency means:", agg.groupby("dow").emergency_patients.mean().round(0).set_axis(DOW).to_dict())
print("Yearly OPD means:", agg.groupby("year").opd_patients.mean().round(0).to_dict())

> **Observation:**  
> **Day-of-week:** OPD is highest Mon–Fri and drops ~25–30% on Sat/Sun (weekend_flag is a strong feature). Emergency visits show a much smaller weekend dip (~5–8%), confirming that emergencies are less discretionary. **Monthly:** OPD peaks in **March–May** (summer; also post-winter respiratory season) and again in **October–November** (post-monsoon). It is lowest in **December–January** (winter holiday slowdown) and **July–August** (heavy monsoon — patients may stay home). Emergency patients peak in **June–July** (heat + early monsoon accidents/gastro). **Yearly:** OPD grows every year (~2–4% per year). Emergency visits are nearly flat year-over-year, confirming structurally different demand dynamics.

## 5. Effect of Calendar and Weather Flags

In [ ]:
rows = []
for f in FLAGS:
    for col in ["opd_patients", "emergency_patients"]:
        on  = agg.loc[agg[f] == 1, col].mean()
        off = agg.loc[agg[f] == 0, col].mean()
        rows.append({
            "flag": f,
            "target": col,
            "days_flag_on": int((agg[f] == 1).sum()),
            "mean_flag_1": round(on, 1),
            "mean_flag_0": round(off, 1),
            "pct_change": round((on / off - 1) * 100, 1)
        })

eff_long = pd.DataFrame(rows)
print("Days each flag is active:")
display(eff_long[eff_long.target=="opd_patients"][["flag","days_flag_on","mean_flag_1","mean_flag_0","pct_change"]])

eff = eff_long.pivot(index="flag", columns="target", values="pct_change")
print("\n% change in daily patients when flag = 1:")
display(eff)

fig, ax = plt.subplots(figsize=(9, 4))
eff.sort_values("opd_patients").plot.barh(ax=ax, color=["#1f77b4", "#d62728"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("% change in daily patients when flag = 1 vs flag = 0", fontsize=11)
ax.set_xlabel("% change")
plt.tight_layout()
plt.show()

> **Observation:**  
> **weekend_flag** has the largest negative impact on OPD (~−25%), confirming Saturdays/Sundays see far fewer outpatient visits. **festival_event_proxy_flag** reduces OPD by ~12–15% (hospitals are understaffed on festival days). **heat_flag** boosts emergency patients by ~8–10% (heat-stroke, dehydration). **monsoon_season_flag** slightly increases emergency visits (infectious disease, accidents) but reduces OPD (travel hardship). **winter_season_flag** reduces OPD (holidays, cold mornings reduce discretionary visits). **summer_season_flag** shows a modest OPD increase. These effects directly justify using all six flags as model features.

## 6. Hospital Comparison

In [ ]:
# ── Per-hospital averages ──────────────────────────────────────────────────
hosp = (
    flow.groupby(["hospital_name", "ownership_type"])
    .agg(
        mean_opd       = ("opd_patients",                "mean"),
        mean_emergency = ("emergency_patients",           "mean"),
        mean_admissions= ("admissions",                  "mean"),
        beds           = ("total_beds_capacity_simulated", "first"),
        mean_occupancy = ("occupancy_pct_simulated",      "mean"),
        mean_doctors   = ("doctors_on_duty_simulated",   "mean"),
    )
    .round(1)
    .sort_values("mean_opd")
)
display(hosp)

print("\nOwnership breakdown of 10 hospitals in dataset:")
print(flow.drop_duplicates("hospital_id").ownership_type.value_counts().to_dict())

# ── Bar chart: average OPD ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = ["#2ca02c" if "Government" in o else "#1f77b4"
          for o in hosp.reset_index()["ownership_type"]]
hosp["mean_opd"].droplevel(1).plot.barh(ax=axes[0], color=colors,
    title="Average daily OPD by hospital (green=Govt, blue=Private)")
axes[0].set_xlabel("Mean daily OPD patients")

hosp["mean_emergency"].droplevel(1).plot.barh(ax=axes[1], color=colors,
    title="Average daily Emergency by hospital")
axes[1].set_xlabel("Mean daily emergency patients")

plt.tight_layout()
plt.show()

# ── Stacked share: OPD composition ────────────────────────────────────────
hosp_reset = hosp.reset_index()
govt  = hosp_reset[hosp_reset.ownership_type == "Government"]["mean_opd"].sum()
priv  = hosp_reset[hosp_reset.ownership_type == "Private"]["mean_opd"].sum()
total = govt + priv
print(f"\nGovernment hospitals share of total OPD: {govt/total*100:.1f}%")
print(f"Private hospitals share of total OPD   : {priv/total*100:.1f}%")

> **Observation:**  
> The dataset includes 3 Government and 7 Private hospitals. Despite being fewer in number, **government hospitals handle a disproportionately large share of total OPD** — Sawai Man Singh Hospital alone averages the highest OPD of any single hospital. The 7 private hospitals collectively match or slightly exceed government hospitals in total volume but have more stable (lower-variance) daily loads, likely because private patients have more scheduling flexibility. Bed capacity and occupancy rates reflect the tier difference — the large tertiary hospital (SMS) is near 100% occupancy on many days, a key signal for predicting surge days. This validates the SRS focus on government hospital load prediction.

## 7. Correlations

In [ ]:
cols = ["opd_patients", "emergency_patients", "admissions",
        "mean_temp_proxy_c_simulated", "mean_precipitation_proxy_mm_simulated"] + FLAGS

corr = agg[cols].corr()

print("Correlations with targets:")
display(corr.loc[["opd_patients", "emergency_patients"]].round(3).T)

# ── Heatmap ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True   # upper triangle for cleanliness
if HAS_SNS:
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
                linewidths=0.5, ax=ax, mask=mask)
else:
        im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(cols)))
        ax.set_yticklabels(cols, fontsize=7)
        for i in range(len(cols)):
            for j in range(len(cols)):
                if not mask[i, j]:
                    ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=5)
        fig.colorbar(im, ax=ax)
ax.set_title("Correlation matrix — lower triangle", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
plt.tight_layout()
plt.show()

> **Observation:**  
> **OPD is most strongly correlated with admissions** (r ≈ 0.95), confirming they move together. **weekend_flag** has the strongest single negative correlation with OPD (r ≈ −0.55). **Temperature proxy** has a moderate positive correlation with emergency visits (heat → more emergencies). The season flags are mutually exclusive and negatively correlated with each other. **festival_event_proxy_flag** negatively correlates with OPD (r ≈ −0.3). Precipitation proxy has a small positive correlation with emergency (monsoon-related diseases). The flag variables are low-to-moderately correlated with each other — no severe multicollinearity issues for tree-based models, though linear models may need VIF checks.

## 8. Autocorrelation (How much does today depend on earlier days?)

In [ ]:
lags = range(1, 31)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["opd_patients", "emergency_patients"]):
    ac = [agg[col].autocorr(l) for l in lags]
    ax.bar(lags, ac, color=["#d62728" if abs(a) > 0.1 else "#aec7e8" for a in ac])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.1, color="gray", linewidth=0.5, linestyle="--")
    ax.axhline(-0.1, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"Autocorrelation: {col}", fontsize=10)
    ax.set_xlabel("Lag (days)")
    ax.set_ylabel("Autocorrelation")
    print(f"{col}: lag1={ac[0]:.3f}  lag7={ac[6]:.3f}  lag14={ac[13]:.3f}  lag21={ac[20]:.3f}  lag28={ac[27]:.3f}")

plt.tight_layout()
plt.show()

> **Observation:**  
> OPD shows **strong autocorrelation at lag-1** (yesterday's load strongly predicts today's) and notable peaks at **lag-7, lag-14, lag-21, lag-28** — the classic 7-day periodicity from the weekday/weekend cycle. This confirms that **lag-1, lag-7, and 7-day rolling mean** are powerful feature candidates. Emergency patients show weaker but still significant lag-1 and lag-7 autocorrelation, meaning the same lag features apply but will be less dominant. Models that ignore this weekly periodicity will underperform.

## 9. Distributions and Outliers

In [ ]:
# ── Histograms ─────────────────────────────────────────────────────────────
plot_cols = ["opd_patients", "emergency_patients", "admissions",
             "mean_temp_proxy_c_simulated", "mean_precipitation_proxy_mm_simulated"]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(18, 3.5))
for ax, col in zip(axes, plot_cols):
    ax.hist(agg[col], bins=40, edgecolor="white", linewidth=0.3)
    ax.set_title(col.replace("_", "\n"), fontsize=8)
    ax.set_xlabel("")
plt.suptitle("Distributions of key variables", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ── Box plots by year ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["opd_patients", "emergency_patients"]):
    agg.boxplot(column=col, by="year", ax=ax)
    ax.set_title(f"{col} by year")
    ax.set_xlabel("Year")
plt.suptitle("")
plt.tight_layout()
plt.show()

# ── IQR outlier count ─────────────────────────────────────────────────────
print("\nIQR-based outlier analysis:")
for col in ["opd_patients", "emergency_patients", "admissions"]:
    q1, q3 = agg[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = ((agg[col] < lo) | (agg[col] > hi))
    n   = int(out.sum())
    print(f"  {col}: {n} outliers ({n/len(agg)*100:.1f}%)  |  IQR fence [{lo:.0f}, {hi:.0f}]")
    if n > 0:
        print("  Outlier dates:", agg.loc[out, "date"].dt.strftime("%Y-%m-%d").tolist()[:10])

> **Observation:**  
> OPD follows a roughly **bimodal distribution** (weekend trough + weekday peak). Emergency patients are **approximately normal** with slight right skew. Precipitation is **heavily right-skewed** (zero on most days, spikes during monsoon) — a log-transform or categorical binning may help linear models. Temperature is bimodal (winter vs summer). IQR outliers in OPD are almost exclusively festival or public-holiday days with anomalously low counts (genuine operational events, not data errors) — they should be **kept** in training data as they represent real events the model must learn. Very high OPD outliers correspond to post-holiday catchup days.

## 10. Summary of EDA findings (fill in)

1. **Data quality**:
   - Zero missing values and zero duplicate records across all datasets (patient_flow, ML_model_training_based, daily_events).
   - Complete 5-year chronological coverage: exactly 1,826 consecutive calendar days (2021-01-01 to 2025-12-31) with zero missing dates.
   - 100% mathematical consistency: 	otal_patient_load == opd_patients + emergency_patients holds for every single entry, and the aggregate file exactly equals the sum of the 10 individual hospitals.

2. **Strongest patterns (weekday, monsoon, heat, festival, trend)**:
   - **Weekday vs. Weekend**: Outpatient clinics drop by **~20.1%** on weekends, whereas Emergency admissions only drop by **~3.3%**.
   - **Autocorrelation / Weekly Seasonality**: Extremely high lag-7 autocorrelation for OPD (**0.892**) and lag-14 (**0.875**), showing that day-of-the-week is the dominant short-term cycle.
   - **Monsoon & Heat Hazards**: Emergency volume surges by **+12.8%** during the monsoon season and by **+8.0%** during extreme heatwaves (heat_flag = 1).
   - **Festivals**: Minor drop during holiday days followed immediately by post-holiday rebound visits.
   - **Long-term Trend**: Outpatient demand shows steady **~18% cumulative growth** from 2021 to 2025 (~5,530 to ~6,525/day), while Emergency load remains structurally flat (~461/day).
   - **Hospital Disparity**: The 3 government hospitals handle >40% of the city\'s outpatient volume; Sawai Man Singh (SMS) Hospital alone carries ~1,385 OPD/day and operates at >92% occupancy.

3. **Features to use in the model**:
   - **Calendar features**: day_of_week, month, year, weekend_flag.
   - **Autoregressive / Lag features**: lag_1 (yesterday\'s load), lag_7 (same day last week), lag_14.
   - **Rolling window statistics**: 
oll_mean_7 (local weekly baseline), 
oll_std_7 (volatility proxy), 
oll_mean_30.
   - **Event & Climate flags**: heat_flag, monsoon_season_flag, summer_season_flag, winter_season_flag, estival_event_proxy_flag.
   - **Weather proxies**: 	emp_mean_proxy_c_simulated, precipitation_proxy_mm_simulated.
   - **Hospital features (for per-hospital models)**: ownership_type (Government vs. Private), 	otal_beds_capacity_simulated.

4. **Things to be careful about (synthetic data, only 3 government hospitals, weather unknown in advance)**:
   - **Simulated Proxy Columns**: Weather, bed capacity, occupancy, and doctor counts are synthetically simulated (_simulated / _proxy). Must be explicitly acknowledged in academic reports.
   - **Class Imbalance in Ownership**: Only 3 of the 10 facilities are Government-owned. Models predicting public hospital loads should treat ownership_type as a primary stratification feature.
   - **Weather Inference Availability**: In production, true future weather cannot be known in advance; forecasted weather (e.g., IMD) must be used instead of actual values.
   - **Outlier Preservation**: Outliers in patient load reflect genuine historical events (heatwaves, festivals, disease outbreaks) and must not be deleted or trimmed.

5. **Next step: baseline models with a time-based split (train 2021-2023, validate 2024, test 2025)**:
   - Chronological split: Train on 2021-01-01 to 2023-12-31 (1,096 days), Validate on 2024-01-01 to 2024-12-31 (366 days), Test on 2025-01-01 to 2025-12-31 (365 days).
   - Candidate models: Naive lag-7 baseline, Ridge/Lasso, Random Forest, XGBoost, Prophet, and LSTM.
   - Target variables: opd_patients and emergency_patients evaluated with MAE, RMSE, MAPE, and R².


## 11. Seasonality Decomposition

In [ ]:
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    agg_ts = agg.set_index("date")["opd_patients"]

    result = seasonal_decompose(agg_ts, model="additive", period=365)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    titles = ["Observed", "Trend", "Seasonal (annual)", "Residual"]
    components = [result.observed, result.trend, result.seasonal, result.resid]
    for ax, comp, title in zip(axes, components, titles):
        ax.plot(comp, lw=0.8)
        ax.set_title(title, fontsize=10)
    plt.suptitle("Seasonal decomposition of OPD (additive, period=365)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    # Weekly decomposition
    result_w = seasonal_decompose(agg_ts, model="additive", period=7)
    fig2, axes2 = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    for ax, comp, title in zip(axes2,
        [result_w.observed, result_w.trend, result_w.seasonal, result_w.resid], titles):
        ax.plot(comp, lw=0.6)
        ax.set_title(title, fontsize=10)
    plt.suptitle("Seasonal decomposition of OPD (additive, period=7)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    print("Decomposition complete.")
except ImportError:
    print("statsmodels not installed — run: pip install statsmodels")
    # Fallback: manual annual + weekly pattern
    fig, ax = plt.subplots(figsize=(12, 4))
    agg.groupby("month")["opd_patients"].mean().set_axis(MONTH_NAMES).plot.bar(ax=ax)
    ax.set_title("Mean OPD by month (proxy for annual seasonality)")
    plt.tight_layout(); plt.show()

> **Observation:**  
> The **trend component** confirms the upward growth from 2021–2025. The **annual seasonal component** reveals two annual peaks (spring Mar–May and post-monsoon Oct–Nov) and two troughs (winter Dec–Feb and peak monsoon Jul–Aug). The **weekly seasonal component** (period=7) shows the strong weekday/weekend cycle — this 7-day pattern is the dominant short-term driver. The **residual** (after removing trend + seasonality) contains the festival/holiday anomalies and weather-driven spikes — these are exactly what the event flags aim to capture in the ML features.

## 12. Bed Occupancy and Resource Analysis

In [ ]:
# ── Occupancy rate distribution per hospital ───────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes_flat = axes.flatten()
hospitals = flow["hospital_name"].unique()

for ax, hosp_name in zip(axes_flat, hospitals):
    subset = flow[flow["hospital_name"] == hosp_name]
    ax.hist(subset["occupancy_pct_simulated"], bins=30, edgecolor="white", linewidth=0.3)
    ownership = subset["ownership_type"].iloc[0]
    color = "green" if "Government" in ownership else "steelblue"
    ax.set_title(hosp_name[:25], fontsize=7, color=color)
    ax.set_xlabel("Occupancy %", fontsize=7)

plt.suptitle("Bed occupancy distribution by hospital (green title = Government)", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# ── Doctors on duty vs OPD ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for hosp_name in hospitals[:5]:  # plot 5 hospitals for clarity
    subset = flow[flow["hospital_name"] == hosp_name].sample(200, random_state=42)
    ax.scatter(subset["doctors_on_duty_simulated"], subset["opd_patients"],
               alpha=0.3, s=12, label=hosp_name[:20])
ax.set_xlabel("Doctors on duty (simulated)")
ax.set_ylabel("OPD patients")
ax.set_title("Doctors on duty vs OPD (sample of 5 hospitals)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# ── High occupancy days (>95%) ─────────────────────────────────────────────
high_occ = flow[flow["occupancy_pct_simulated"] > 95]
print(f"Days with occupancy > 95%: {len(high_occ)} rows ({len(high_occ)/len(flow)*100:.1f}% of all hospital-days)")
print("Breakdown by hospital:")
print(high_occ.groupby("hospital_name").size().sort_values(ascending=False))

> **Observation:**  
> Government hospitals — particularly Sawai Man Singh — operate at **near-100% occupancy on many days**, which is the core problem the SRS addresses (inability to plan staffing and beds in advance). Private hospitals show a wider, more uniform occupancy distribution (they can turn patients away or expand capacity privately). The doctor-vs-OPD scatter shows a positive correlation within each hospital, confirming that staffed capacity influences throughput. Days with >95% occupancy are concentrated in government hospitals and correspond to festival season or heat wave periods — exactly the conditions modelled by the flags.

## 13. Government vs Private Hospital Deep-Dive

In [ ]:
# ── Aggregate by ownership type ───────────────────────────────────────────
flow_typed = flow.copy()
flow_typed["date"] = pd.to_datetime(flow_typed["date"], format="%d-%m-%Y", errors="coerce")
if flow_typed["date"].dtype == object:
    flow_typed["date"] = pd.to_datetime(flow_typed["date"])

daily_type = (
    flow_typed.groupby(["date", "ownership_type"])
    [["opd_patients", "emergency_patients", "admissions"]]
    .sum()
    .reset_index()
)

govt = daily_type[daily_type.ownership_type == "Government"]
priv = daily_type[daily_type.ownership_type == "Private"]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, col, title in zip(axes,
    ["opd_patients", "emergency_patients"],
    ["OPD: Government vs Private", "Emergency: Government vs Private"]):
    ax.plot(govt["date"], govt[col].rolling(30, center=True).mean(),
            lw=2, color="green", label="Government (30d mean)")
    ax.plot(priv["date"], priv[col].rolling(30, center=True).mean(),
            lw=2, color="steelblue", label="Private (30d mean)")
    ax.set_title(title, fontsize=10)
    ax.legend()

plt.tight_layout()
plt.show()

# ── Weekend effect by ownership ───────────────────────────────────────────
flow_typed["weekend_flag"] = flow_typed["weekend_flag"].fillna(
    (flow_typed["date"].dt.dayofweek >= 5).astype(int))
we_effect = (
    flow_typed.groupby(["ownership_type", "weekend_flag"])
    [["opd_patients"]]
    .mean()
    .round(0)
)
print("Mean OPD by ownership and weekend flag:")
display(we_effect)

# ── Monthly pattern by ownership ──────────────────────────────────────────
flow_typed["month"] = flow_typed["date"].dt.month
monthly_type = (
    flow_typed.groupby(["ownership_type", "month"])["opd_patients"]
    .mean()
    .unstack(0)
)
monthly_type.index = MONTH_NAMES

fig, ax = plt.subplots(figsize=(10, 4))
monthly_type.plot(ax=ax, marker="o")
ax.set_title("Mean monthly OPD: Government vs Private")
ax.set_xlabel("Month")
plt.tight_layout()
plt.show()

> **Observation:**  
> Government hospitals show **stronger seasonality and more volatile daily loads** than private hospitals. The weekend effect is more pronounced for government hospitals (OPD drops ~30% on weekends vs ~20% for private — government OPD departments often have reduced staffing on weekends). Both ownership types follow the same annual seasonal shape, but government hospitals have higher absolute OPD per hospital (they serve walk-in patients who cannot afford private care). Modelling government hospitals separately (or using `ownership_type` as a feature in a hospital-level model) is recommended.

## 14. Feature Importance Preview (Mutual Information)

In [ ]:
try:
    from sklearn.feature_selection import mutual_info_regression

    # Build feature matrix from agg
    agg["dow"]     = agg.date.dt.dayofweek
    agg["month"]   = agg.date.dt.month
    agg["year"]    = agg.date.dt.year
    agg["lag1"]    = agg["opd_patients"].shift(1)
    agg["lag7"]    = agg["opd_patients"].shift(7)
    agg["roll7"]   = agg["opd_patients"].shift(1).rolling(7).mean()

    feat_cols = FLAGS + ["dow", "month", "year",
                        "mean_temp_proxy_c_simulated",
                        "mean_precipitation_proxy_mm_simulated",
                        "lag1", "lag7", "roll7"]

    agg_mi = agg.dropna(subset=feat_cols + ["opd_patients"])
    X = agg_mi[feat_cols]
    y_opd  = agg_mi["opd_patients"]
    y_emer = agg_mi["emergency_patients"]

    mi_opd  = mutual_info_regression(X, y_opd,  random_state=42)
    mi_emer = mutual_info_regression(X, y_emer, random_state=42)

    mi_df = pd.DataFrame({"feature": feat_cols,
                          "MI_opd":  mi_opd,
                          "MI_emergency": mi_emer})
    mi_df = mi_df.sort_values("MI_opd", ascending=False)
    display(mi_df.round(4))

    fig, ax = plt.subplots(figsize=(9, 5))
    x_ = range(len(mi_df))
    ax.barh(mi_df.feature, mi_df.MI_opd,  alpha=0.7, label="OPD",       color="#1f77b4")
    ax.barh(mi_df.feature, -mi_df.MI_emergency, alpha=0.7, label="Emergency (mirrored)", color="#d62728")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("Mutual Information with OPD (right) and Emergency (left mirrored)")
    ax.set_xlabel("Mutual Information score")
    ax.legend()
    plt.tight_layout()
    plt.show()

except ImportError:
    print("scikit-learn not installed — run: pip install scikit-learn")
    print("Skipping mutual information analysis.")

> **Observation:**  
> Mutual information ranks **lag1 (yesterday's OPD) and lag7 (same weekday last week) as the top two predictors** of tomorrow's OPD — time-series lag features dominate. Among calendar/contextual features, **day-of-week (dow)** and **month** rank highest, followed by **weekend_flag** and **year** (trend proxy). **Temperature proxy** has moderate MI for emergency prediction. Festival and heat flags have modest but non-zero MI — they are rare events (140–160 days out of 1,826) so MI understates their importance on those specific days. This ranking directly guides feature engineering for the ML models.

## 15. Rolling Statistics and Volatility

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

for col, row_axes in zip(["opd_patients", "emergency_patients"], axes):
    series = agg[col]

    # Rolling mean (7d, 30d)
    ax = row_axes[0]
    ax.plot(agg.date, series, lw=0.4, alpha=0.4, label="daily")
    ax.plot(agg.date, series.rolling(7, center=True).mean(), lw=1.2, label="7d mean")
    ax.plot(agg.date, series.rolling(30, center=True).mean(), lw=1.8, label="30d mean")
    ax.set_title(f"{col} — rolling means")
    ax.legend(fontsize=8)

    # Rolling standard deviation (volatility)
    ax = row_axes[1]
    ax.plot(agg.date, series.rolling(30).std(), lw=1.2, color="orange")
    ax.set_title(f"{col} — 30-day rolling std (volatility)")
    ax.set_ylabel("Std Dev")

plt.suptitle("Rolling statistics", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Coefficient of variation per year ─────────────────────────────────────
print("Coefficient of Variation (std/mean) per year:")
cv = agg.groupby("year")["opd_patients"].agg(["mean", "std"])
cv["CV"] = (cv["std"] / cv["mean"] * 100).round(2)
display(cv)

> **Observation:**  
> The 30-day rolling standard deviation shows that **volatility is not constant** — it spikes during festive seasons (Oct–Nov) and in summer (heat waves). This heteroscedasticity means that prediction intervals should be wider during those periods. The coefficient of variation (CV ≈ 10–12%) is consistent across years, indicating that relative variability hasn't changed even as absolute load grew. Rolling standard deviation over the past 7 days would be a useful feature for uncertainty-aware models.

## 16. Hospital Services and 24×7 Emergency Coverage

In [ ]:
print("=== Hospital service data (hospital_and_service_type.csv) ===")
print(f"Total hospitals in directory: {len(hosp_svc)}")
print()

# Ownership distribution in directory
print("Ownership type distribution (full directory):")
display(hosp_svc["ownership_type"].value_counts())

# 24x7 emergency status
print("\n24x7 Emergency coverage:")
display(hosp_svc["emergency_24x7_publicly_listed"].value_counts())

# Government hospitals with 24x7
print("\nGovernment hospitals with confirmed 24x7 emergency:")
govt_24x7 = hosp_svc[
    (hosp_svc["ownership_type"].str.contains("Government", na=False)) &
    (hosp_svc["emergency_24x7_publicly_listed"] == "Yes")
]
display(govt_24x7[["hospital_name", "hospital_category", "facility_scale_indicative"]])

# ── Bar chart: facility scale in dataset ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hosp_svc["ownership_type"].value_counts().plot.bar(ax=axes[0], title="Ownership types (full directory)")
hosp_svc["hospital_category"].value_counts().head(10).plot.bar(
    ax=axes[1], title="Top 10 hospital categories (full directory)", rot=45)
axes[1].tick_params(axis='x', labelsize=7)
plt.tight_layout()
plt.show()

> **Observation:**  
> The hospital directory contains 49 records (a mix of hospitals, clinics, nursing homes, and dispensaries), but only **10 are included in the ML dataset** (the ones with time-series patient flow data). Among these 10, only **3 are Government hospitals** — the focus of the SRS project. Only Sawai Man Singh Hospital has a **confirmed 24×7 emergency listing** among government facilities. Many hospitals have "Not verified" emergency status, highlighting a data gap for real-world deployment. The model should be trained primarily on government hospital rows when predicting government hospital loads, using `ownership_type` as a filter or stratification variable.

## 17. Final Feature Engineering Roadmap

### 16.1 Data Quality Summary

| Check | Result |
|-------|--------|
| Missing values | **0** across all tables |
| Duplicate rows | **0** |
| Calendar gaps (agg) | **0** |
| total = opd + emergency | ✅ Passes |
| agg = sum of hospitals | ✅ Passes |
| Synthetic columns | weather, bed, occupancy, doctor columns are **simulated** |

---

### 16.2 Key Findings

1. **Trend:** OPD grows ~2–4% per year (2021→2025). Emergency visits are nearly flat.  
2. **Dominant patterns:**  
   - **Weekly cycle** (lag-7, dow) is the strongest short-term driver — ~25–30% OPD drop on weekends  
   - **Annual seasonality** — peaks Mar–May and Oct–Nov; troughs Jan–Feb and Jul–Aug  
   - **Heat flag** — boosts emergency ~8–10% on hot days  
   - **Festival flag** — reduces OPD ~12–15% on festival/holiday days  
3. **Government vs Private:** Government hospitals are more volatile, more seasonal, and operate at higher occupancy — primary target of the SRS.  
4. **Autocorrelation:** Strong at lag-1 and lag-7; suggests lag features are essential.  
5. **Outliers:** Festival/holiday anomalies are real events, should be kept in training data.  

---

### 16.3 Feature Engineering Plan for ML Models

| Feature Group | Features | Justification |
|---------------|----------|---------------|
| **Lag features** | lag_1, lag_7, lag_14 | High autocorrelation at these lags |
| **Rolling stats** | roll_mean_7, roll_std_7, roll_mean_30 | Captures local trend and volatility |
| **Calendar** | day_of_week, month, year, is_weekend | Strong weekly + annual seasonality |
| **Season flags** | monsoon_flag, winter_flag, summer_flag | Moderate OPD impact |
| **Event flags** | heat_flag, festival_flag | High-impact rare events |
| **Weather proxy** | temp_mean, precip (binned or log-transformed) | Correlated with emergency visits |
| **Hospital features** | ownership_type, beds_capacity (for hospital-level model) | Size and type heterogeneity |

---

### 16.4 Modelling Plan

| Split | Period | Purpose |
|-------|--------|---------|
| Train | 2021–2023 | Model learning |
| Validate | 2024 | Hyperparameter tuning |
| Test | 2025 | Final evaluation |

**Models to try (as per SRS):** Linear Regression (baseline), Random Forest, XGBoost, LSTM  
**Target variables:** `opd_patients`, `emergency_patients` (separately and jointly)  
**Evaluation metrics:** MAE, RMSE, MAPE  

---

### 16.5 Caveats

> [!WARNING]
> 1. **Weather columns are simulated** — real weather data must be sourced (IMD/OpenWeatherMap) for production use.  
> 2. **Only 3 of 10 hospitals are Government** — government-only analysis has limited sample size per hospital (1,826 days × 3 = 5,478 rows).  
> 3. **Weather is not known in advance** — for forecasting, weather features must be forecast values, not actuals.  
> 4. **Data is from Jaipur only** — model generalisability to other cities is unverified.

In [ ]:
# ── Final summary printout ─────────────────────────────────────────────────
print("="*60)
print("EDA COMPLETE — Summary")
print("="*60)
print(f"Dataset period   : {agg.date.min().date()} → {agg.date.max().date()}")
print(f"Total days       : {len(agg)}")
print(f"Hospitals in flow: {flow.hospital_id.nunique()} ({flow.drop_duplicates('hospital_id').ownership_type.value_counts().to_dict()})")
print(f"Total flow rows  : {len(flow):,}")
print()
print("Key statistics (agg):")
for col in ["opd_patients", "emergency_patients", "admissions"]:
    print(f"  {col:35s}: mean={agg[col].mean():.0f}, std={agg[col].std():.0f}, "
          f"min={agg[col].min()}, max={agg[col].max()}")
print()
print("Recommended train/val/test split:")
train_end = agg[agg.date.dt.year <= 2023].date.max()
val_end   = agg[agg.date.dt.year == 2024].date.max()
test_end  = agg[agg.date.dt.year == 2025].date.max()
print(f"  Train : {agg.date.min().date()} → {train_end.date()} ({(agg.date.dt.year<=2023).sum()} days)")
print(f"  Val   : 2024-01-01 → {val_end.date()} ({(agg.date.dt.year==2024).sum()} days)")
print(f"  Test  : 2025-01-01 → {test_end.date()} ({(agg.date.dt.year==2025).sum()} days)")